In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader
import random
from glob import glob
import cv2
import numpy as np

In [2]:
image_paths = glob("../data/**/*.jpg", recursive=True)

images = [cv2.imread(p) for p in image_paths if cv2.imread(p) is not None]

print(len(images))

26


In [3]:
gray_images = []
for img in images:
    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    else:
        gray = img 
    
    gray_images.append(gray)

In [ ]:
class SiameseDataset(Dataset):
    def __init__(self, images, transform=None):
        self.images = images
        self.transform = transform
        self.n = len(images)

    def augment(self, img):
        if np.random.rand() > 0.5:
            img = cv2.flip(img, 1)
    
        if np.random.rand() > 0.5:
            noise = np.random.normal(0, 5, img.shape)
            img = img + noise
            img = np.clip(img, 0, 255)
    
        return img.astype("uint8")
    
    def __len__(self):
        return self.n 
    
    def __getitem__(self, idx):
        img1 = self.images[idx].copy()

        if random.random() > 0.5:
            # positive pair
            img2 = self.augment(img1)
            label = 1
        else:
            idx2 = random.randint(0, self.n - 1)
            while idx2 == idx:
                idx2 = random.randint(0, self.n - 1)
            
            img2 = self.images[idx2]
            label = 0

        img1 = img1 / 255.0
        img2 = img2 / 255.0

        img1 = torch.tensor(img1, dtype=torch.float32).unsqueeze(0)
        img2 = torch.tensor(img2, dtype=torch.float32).unsqueeze(0)

        label = torch.tensor(label, dtype=torch.float32)

        return img1, img2, label


In [5]:
dataset = SiameseDataset(gray_images)

loader = DataLoader(dataset, batch_size=8, shuffle=True)

In [6]:
class EmbeddingNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.fc = nn.Sequential(
            nn.Linear(64 * 5 * 5, 128)
        )

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        x = F.normalize(x, p=2, dim=1) 
        return x

In [7]:
class SiameseNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = EmbeddingNet()

    def forward(self, x1, x2):
        out1 = self.network(x1)
        out2 = self.network(x2)
        return out1, out2

In [8]:
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=0.5):
        super().__init__()
        self.margin = margin

    def forward(self, out1, out2, label):

        cos_sim = F.cosine_similarity(out1, out2)

        loss = torch.mean(
            label * (1 - cos_sim) +
            (1 - label) * torch.clamp(cos_sim - self.margin, min=0)
        )

        return loss

In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = SiameseNet().to(device)
criterion = ContrastiveLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(5):
    total_loss = 0

    loop = tqdm(loader, desc=f"Epoch {epoch}")

    for img1, img2, label in loop:
        img1, img2, label = img1.to(device), img2.to(device), label.to(device)

        optimizer.zero_grad()

        out1, out2 = model(img1, img2)
        loss = criterion(out1, out2, label)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        
        loop.set_postfix(loss=loss.item())

    print(f"Epoch {epoch} Avg Loss: {total_loss / len(loader):.4f}")

Epoch 0:   0%|          | 0/4 [01:56<?, ?it/s]


KeyboardInterrupt: 